# 0. Rename RHS Folders by Stim Waveform

Use this first utility block when you have a parent stimulation folder containing multiple RHS session folders. Paste the parent folder path, click **Preview Rename Plan**, and review the proposed names. Click **Apply Renames** only when the preview looks correct.

The suffix is decoded from the first recorded channel in each RHS folder that contains nonzero `stim_data`, so it can handle recordings with multiple amplifier channels and stimulation on only one channel. The folder name suffix is formatted like `cathodic first 100us 200uA 1Hz 30 pulses 999ms RP comp`.


### Helper code used by Block 0

The notebook is the user interface. The folder scanning, waveform decoding, and actual renaming live in [`rename_rhs_folders_by_stim_waveform.py`](rename_rhs_folders_by_stim_waveform.py).

- **Preview Rename Plan** reads each immediate RHS session folder and shows the old/new folder names without changing anything.
- **Apply Renames** applies the previewed plan only for folders marked `rename`.
- If you paste a single RHS session folder instead of its parent, the utility will rename that one folder.

- The suffix also adds `999ms RP` from the inferred refractory gap and `comp` when the RHS compliance-limit bit is present.


In [ ]:
from __future__ import annotations

from pathlib import Path
import html

import ipywidgets as widgets
from IPython.display import HTML, display

# -----------------------------------------------------------------------------
# 1) Import the RHS folder rename helper functions.
# -----------------------------------------------------------------------------
from rename_rhs_folders_by_stim_waveform import apply_rename_plan, plan_renames


# -----------------------------------------------------------------------------
# 2) Choose the default parent folder that contains RHS session folders.
# -----------------------------------------------------------------------------
# This path only seeds the UI text box. Paste any parent folder from your computer
# before clicking Preview Rename Plan.
DEFAULT_RENAME_PARENT_DIR = Path(
    "/Users/jf/Library/CloudStorage/SynologyDrive-Endovascular/Jill/"
    "20260624 pt stimulation"
)


def rename_folder_path_from_text(value: str) -> Path:
    """Clean a pasted folder path from the widget and return an absolute Path."""
    cleaned = value.strip().strip('"').strip("'")
    cleaned = cleaned.replace("\r", "").replace("\n", "")
    return Path(cleaned).expanduser().resolve()


# -----------------------------------------------------------------------------
# 3) Build the rename controls.
# -----------------------------------------------------------------------------
rename_parent_text = widgets.Textarea(
    value=str(DEFAULT_RENAME_PARENT_DIR),
    description="Parent folder",
    placeholder="Paste parent folder containing RHS session folders here",
    continuous_update=False,
    layout=widgets.Layout(width="100%", height="46px"),
    style={"description_width": "105px"},
)
preview_rename_button = widgets.Button(
    description="Preview Rename Plan",
    icon="search",
    button_style="info",
    layout=widgets.Layout(width="190px"),
)
apply_rename_button = widgets.Button(
    description="Apply Renames",
    icon="check",
    button_style="warning",
    disabled=True,
    layout=widgets.Layout(width="160px"),
)
rename_status = widgets.HTML()
rename_output = widgets.Output()
rename_state = {"plans": []}


# -----------------------------------------------------------------------------
# 4) Render the preview/apply results as an embedded table.
# -----------------------------------------------------------------------------
def rename_plan_table(plans) -> HTML:
    """Return an HTML table summarizing the current rename plan."""
    rows = []
    for plan in plans:
        source_name = html.escape(plan.source.name)
        target_name = "" if plan.target is None else html.escape(plan.target.name)
        stim_channel = ""
        suffix = ""
        if plan.summary is not None:
            stim_channel = html.escape(plan.summary.channel_name)
            suffix = html.escape(plan.summary.suffix)
        message = html.escape(plan.error or "")
        color = {
            "rename": "#0b6bcb",
            "renamed": "#1b7f2a",
            "already named": "#555555",
            "skip": "#8a6d00",
            "error": "#b00020",
        }.get(plan.status, "#333333")
        rows.append(
            "<tr>"
            f"<td style='padding:4px 8px;color:{color};font-weight:600'>{html.escape(plan.status)}</td>"
            f"<td style='padding:4px 8px'>{source_name}</td>"
            f"<td style='padding:4px 8px'>{target_name}</td>"
            f"<td style='padding:4px 8px'>{stim_channel}</td>"
            f"<td style='padding:4px 8px'>{suffix}</td>"
            f"<td style='padding:4px 8px;color:#b00020'>{message}</td>"
            "</tr>"
        )
    table = (
        "<table style='border-collapse:collapse;font-size:13px'>"
        "<thead><tr>"
        "<th style='text-align:left;padding:4px 8px'>Status</th>"
        "<th style='text-align:left;padding:4px 8px'>Current folder</th>"
        "<th style='text-align:left;padding:4px 8px'>New folder</th>"
        "<th style='text-align:left;padding:4px 8px'>Stim channel</th>"
        "<th style='text-align:left;padding:4px 8px'>Waveform suffix</th>"
        "<th style='text-align:left;padding:4px 8px'>Message</th>"
        "</tr></thead><tbody>"
        + "".join(rows)
        + "</tbody></table>"
    )
    return HTML(table)


def preview_rename_plan(_button=None) -> None:
    """Scan the selected folder and show proposed names without renaming."""
    apply_rename_button.disabled = True
    with rename_output:
        rename_output.clear_output(wait=True)
        try:
            parent_folder = rename_folder_path_from_text(rename_parent_text.value)
            plans = plan_renames(parent_folder)
        except Exception as exc:
            rename_state["plans"] = []
            rename_status.value = f"<b style='color:#b00020'>Preview failed:</b> {html.escape(str(exc))}"
            return

        rename_state["plans"] = plans
        rename_count = sum(1 for plan in plans if plan.will_rename)
        error_count = sum(1 for plan in plans if plan.status == "error")
        apply_rename_button.disabled = rename_count == 0
        rename_status.value = (
            f"Previewed <b>{len(plans)}</b> RHS folder(s). "
            f"Folders to rename: <b>{rename_count}</b>. Errors: <b>{error_count}</b>."
        )
        display(rename_plan_table(plans))


def apply_renames(_button=None) -> None:
    """Apply the last previewed plan. Preview again after changing the path."""
    plans = rename_state.get("plans", [])
    if not plans:
        preview_rename_plan()
        plans = rename_state.get("plans", [])
    if not plans:
        return

    with rename_output:
        rename_output.clear_output(wait=True)
        applied = apply_rename_plan(plans)
        rename_state["plans"] = applied
        renamed_count = sum(1 for plan in applied if plan.status == "renamed")
        error_count = sum(1 for plan in applied if plan.status == "error")
        apply_rename_button.disabled = True
        rename_status.value = (
            f"Applied rename plan. Renamed: <b>{renamed_count}</b>. "
            f"Errors: <b>{error_count}</b>."
        )
        display(rename_plan_table(applied))


preview_rename_button.on_click(preview_rename_plan)
apply_rename_button.on_click(apply_renames)

# Display the full Function 0 interface inline in VS Code/Jupyter.
display(
    widgets.VBox(
        [
            rename_parent_text,
            widgets.HBox([preview_rename_button, apply_rename_button, rename_status]),
            rename_output,
        ]
    )
)


# 1. Plot All Channel Data Wideband

Use this first block to generate the raw wideband preview inline in VS Code/Jupyter. Choose or paste the RHS data folder, choose channels and an optional time window, click **Generate Preview**, then click **Save PNG** only if you want to save the image into that same selected data folder.

To change the default folder that appears when this notebook starts, edit `DEFAULT_DATA_DIR` near the top of the code cell below.

### Helper code used by Block 1

The notebook is the user interface. The low-level RHS parsing and raw plot drawing live in [`plot_rhs_raw_wideband_with_stim_legend.py`](plot_rhs_raw_wideband_with_stim_legend.py).

- `parse_channel_selection(...)` accepts entries like `A-014`, `A-014-16`, or `A-014, A-016`.
- `parse_time_window(...)` accepts `all` or entries like `0-10 s` to display only part of the recording.
- `read_rhs_folder(folder, channel)` reads and concatenates all `.rhs` files for each selected channel.
- Raw wideband amplifier samples are converted to microvolts with Intan scaling: `0.195 * (uint16 - 32768)`.
- RHS `stim_data` is decoded from the Intan stim bitfield into signed command current in `uA`.
- No LFP/band-pass filter is applied in Block 1. `MAX_POINTS` only controls the display envelope used to draw long raw traces efficiently.

In [ ]:
from __future__ import annotations

import os
import tempfile
from io import BytesIO
from pathlib import Path

# -----------------------------------------------------------------------------
# 1) Configure notebook output so plots are embedded, not pop-up windows.
# -----------------------------------------------------------------------------
_cache_root = Path(tempfile.gettempdir()) / "codex_matplotlib_cache"
_mpl_cache = _cache_root / "mpl"
_xdg_cache = _cache_root / "xdg"
_mpl_cache.mkdir(parents=True, exist_ok=True)
_xdg_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_mpl_cache))
os.environ.setdefault("XDG_CACHE_HOME", str(_xdg_cache))
os.environ["MPLBACKEND"] = "Agg"

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# -----------------------------------------------------------------------------
# 2) Import the raw wideband helper functions.
# -----------------------------------------------------------------------------
from plot_rhs_raw_wideband_with_stim_legend import (
    channel_selection_label,
    default_output_path,
    find_pulse_segments,
    find_stim_channel_in_data,
    parse_channel_selection,
    parse_time_window,
    plot_raw_channels_with_stim_pulse,
    plot_raw_with_stim_pulse,
    read_rhs_folder,
    sample_slice_for_time_window,
    time_window_label,
)


# -----------------------------------------------------------------------------
# 3) Choose the default data folder and raw plot settings.
# -----------------------------------------------------------------------------
# This path only seeds the UI text box. You can paste a different folder path
# into the RHS folder field before clicking Generate Preview.
DEFAULT_DATA_DIR = Path(
    "/Users/jf/Library/CloudStorage/SynologyDrive-Endovascular/Jill/"
    "20260617 stimulation Pt/"
    "1_260617_180931 anodic-lead 100us 50uA 5 pulses 1000Hz"
)
DEFAULT_CHANNELS = "A-014"
DEFAULT_TIME_WINDOW = "all"
DEFAULT_MAX_POINTS = 600_000
DEFAULT_PULSE_NUMBER = 1


def folder_path_from_text(value: str) -> Path:
    """Clean a pasted folder path from the widget and return an absolute Path."""
    cleaned = value.strip().strip('"').strip("'")
    cleaned = cleaned.replace("\r", "").replace("\n", "")
    return Path(cleaned).expanduser().resolve()


# -----------------------------------------------------------------------------
# 4) Build the raw wideband controls.
# -----------------------------------------------------------------------------
raw_folder_text = widgets.Textarea(
    value=str(DEFAULT_DATA_DIR),
    description="RHS folder",
    placeholder="Paste RHS data folder path here",
    continuous_update=False,
    layout=widgets.Layout(width="100%", height="46px"),
    style={"description_width": "90px"},
)
raw_channel_text = widgets.Text(
    value=DEFAULT_CHANNELS,
    description="Channels",
    placeholder="A-014 or A-014-16",
    layout=widgets.Layout(width="240px"),
    style={"description_width": "75px"},
)
raw_time_text = widgets.Text(
    value=DEFAULT_TIME_WINDOW,
    description="Time (s)",
    placeholder="all or 0-10",
    layout=widgets.Layout(width="190px"),
    style={"description_width": "70px"},
)
raw_max_points_int = widgets.IntText(
    value=DEFAULT_MAX_POINTS,
    description="Max points",
    layout=widgets.Layout(width="210px"),
    style={"description_width": "90px"},
)
raw_pulse_number_int = widgets.IntText(
    value=DEFAULT_PULSE_NUMBER,
    description="Pulse",
    layout=widgets.Layout(width="150px"),
    style={"description_width": "55px"},
)
raw_generate_button = widgets.Button(
    description="Generate Preview",
    icon="play",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)
raw_save_button = widgets.Button(
    description="Save PNG",
    icon="save",
    button_style="success",
    disabled=True,
    layout=widgets.Layout(width="110px"),
)
raw_status = widgets.HTML(value="Choose an RHS folder, then click <b>Generate Preview</b>.")
raw_target_label = widgets.HTML(value="<b>Target:</b> no preview generated yet")
raw_preview_output = widgets.Output()
raw_current_preview = {"png": None, "output_path": None}

display(
    widgets.VBox(
        [
            raw_folder_text,
            widgets.HBox([raw_channel_text, raw_time_text, raw_max_points_int, raw_pulse_number_int]),
            widgets.HBox([raw_generate_button, raw_save_button, raw_target_label]),
            raw_status,
            raw_preview_output,
        ]
    )
)


# -----------------------------------------------------------------------------
# 5) Generate and save the raw wideband preview.
# -----------------------------------------------------------------------------
def generate_raw_preview(_button=None) -> None:
    """Read selected RHS folder and show the raw wideband preview inline."""
    raw_save_button.disabled = True
    raw_current_preview["png"] = None
    raw_current_preview["output_path"] = None
    raw_target_label.value = "<b>Target:</b> no preview generated yet"
    raw_preview_output.clear_output()

    data_folder = folder_path_from_text(raw_folder_text.value)
    max_points = int(raw_max_points_int.value)
    pulse_number = int(raw_pulse_number_int.value)
    try:
        channels = parse_channel_selection(raw_channel_text.value)
        time_window = parse_time_window(raw_time_text.value)
    except ValueError as exc:
        raw_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return
    output_label = channel_selection_label(channels)
    if time_window is not None:
        output_label = f"{output_label}_{time_window_label(time_window)}"
    output_path = default_output_path(data_folder, output_label)

    if not data_folder.exists():
        raw_status.value = f"<b style='color:#b00020'>Folder not found:</b> {data_folder}"
        return
    if not list(data_folder.glob("*.rhs")):
        raw_status.value = f"<b style='color:#b00020'>No .rhs files found in:</b> {data_folder}"
        return

    raw_status.value = f"Reading RHS files from <b>{data_folder}</b>..."
    channel_data = []
    sample_rate_hz = None
    loaded = None
    try:
        for channel in channels:
            raw_uV, stim_uA, channel_sample_rate_hz, channel_loaded = read_rhs_folder(data_folder, channel)
            if sample_rate_hz is not None and channel_sample_rate_hz != sample_rate_hz:
                raise ValueError("Selected channels have different sample rates.")
            sample_rate_hz = channel_sample_rate_hz
            loaded = channel_loaded
            channel_data.append((channel, raw_uV, stim_uA))
    except (FileNotFoundError, ValueError) as exc:
        raw_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    try:
        fig = plot_raw_channels_with_stim_pulse(
            channel_data=channel_data,
            sample_rate_hz=sample_rate_hz,
            folder=data_folder,
            output_path=output_path,
            max_points=max_points,
            pulse_number=pulse_number,
            time_window=time_window,
        )
    except ValueError as exc:
        raw_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    preview_buffer = BytesIO()
    fig.savefig(preview_buffer, format="png", dpi=220)
    plt.close(fig)
    preview_png = preview_buffer.getvalue()

    raw_current_preview["png"] = preview_png
    raw_current_preview["output_path"] = output_path
    raw_save_button.disabled = False
    raw_target_label.value = f"<b>Target:</b> {output_path}"

    first_channel_name, first_raw_uV, first_stim_uA = channel_data[0]
    display_slice, display_bounds = sample_slice_for_time_window(first_raw_uV.size, sample_rate_hz, time_window)
    displayed_samples = display_slice.stop - display_slice.start
    time_status = "all time" if time_window is None else f"{display_bounds[0]:g}-{display_bounds[1]:g} s"
    stim_channel_info = find_stim_channel_in_data(channel_data, slice(0, first_raw_uV.size))
    if stim_channel_info is None:
        stim_status = "no nonzero stim_data found in selected channels"
    else:
        stim_channel_name, _display_stim_uA, pulse_segments = stim_channel_info
        stim_status = f"{len(pulse_segments)} stim pulses on {stim_channel_name} *"
    total_timestamp_gaps = sum(item.timestamp_gaps for item in loaded)
    raw_status.value = (
        f"Loaded {len(loaded)} RHS file(s) for {len(channels)} channel(s), "
        f"displaying {displayed_samples} of {first_raw_uV.size} samples/channel "
        f"({time_status}), "
        f"{stim_status}, "
        f"{total_timestamp_gaps} timestamp gaps."
    )

    with raw_preview_output:
        display(widgets.Image(value=preview_png, format="png", layout=widgets.Layout(width="100%")))


def save_raw_png(_button=None) -> None:
    """Save the current raw wideband preview into the selected RHS folder only."""
    preview_png = raw_current_preview.get("png")
    output_path = raw_current_preview.get("output_path")
    if preview_png is None or output_path is None:
        raw_status.value = "<b style='color:#b00020'>Generate a preview before saving.</b>"
        return
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = output_path.with_name(f".{output_path.stem}.tmp{output_path.suffix}")
    temp_path.write_bytes(preview_png)
    os.replace(temp_path, output_path)
    raw_target_label.value = f"<b>Saved:</b> {output_path}"
    raw_status.value = f"Saved PNG inside selected data folder: <b>{output_path.parent}</b>"


raw_generate_button.on_click(generate_raw_preview)
raw_save_button.on_click(save_raw_png)


# 2. Plot Bandpass Filtered Data

Use this second block to generate a frequency-selected plot from the same raw RHS data. Enter `all` to keep all recorded frequencies, or enter a bandpass range such as `200-400 Hz` or `400-6000 Hz`. You can also enter a signed amplitude window such as `-100 - 100 uV`, and an optional time window such as `10-20 s`.

Only samples inside the signed amplitude window and selected time window are shown, and the y-axis is zoomed to that same amplitude window. The preview is not saved until you click **Save PNG**.

### Helper code used by Block 2

This block uses [`plot_rhs_filtered_wideband.py`](plot_rhs_filtered_wideband.py) for the filter-specific work, while still using the channel parser and RHS reader from `plot_rhs_raw_wideband_with_stim_legend.py`.

- `parse_frequency_range(...)` parses `all` for all recorded frequencies, or entries like `200-400 Hz` for the bandpass filter.
- `parse_amplitude_range(...)` parses signed entries like `-100 - 100 uV` for the amplitude window and y-axis range.
- `parse_channel_selection(...)` accepts entries like `A-014`, `A-014-16`, or `A-014, A-016`.
- `parse_time_window(...)` accepts `all` or entries like `10-20 s` for the displayed time window.
- `bandpass_filter_wideband(...)` keeps all recorded frequencies when Bandpass is `all`, or applies a zero-phase Butterworth bandpass for numeric ranges.
- `plot_filtered_wideband(...)` draws only the bandpass filtered samples inside the selected signed amplitude window and zooms the y-axis to that window.

In [ ]:
from __future__ import annotations

import os
import tempfile
from io import BytesIO
from pathlib import Path

# -----------------------------------------------------------------------------
# 1) Configure notebook output so plots are embedded, not pop-up windows.
# -----------------------------------------------------------------------------
_cache_root = Path(tempfile.gettempdir()) / "codex_matplotlib_cache"
_mpl_cache = _cache_root / "mpl"
_xdg_cache = _cache_root / "xdg"
_mpl_cache.mkdir(parents=True, exist_ok=True)
_xdg_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_mpl_cache))
os.environ.setdefault("XDG_CACHE_HOME", str(_xdg_cache))
os.environ["MPLBACKEND"] = "Agg"

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# -----------------------------------------------------------------------------
# 2) Import RHS reader and filtered-plot helper functions.
# -----------------------------------------------------------------------------
from plot_rhs_raw_wideband_with_stim_legend import (
    channel_selection_label,
    find_stim_channel_in_data,
    parse_channel_selection,
    parse_time_window,
    read_rhs_folder,
    time_window_label,
)
from plot_rhs_filtered_wideband import (
    default_filtered_output_path,
    format_bandpass_status,
    parse_amplitude_range,
    parse_frequency_range,
    plot_filtered_channels,
    plot_filtered_wideband,
)


# -----------------------------------------------------------------------------
# 3) Choose the default data folder and filtered plot settings.
# -----------------------------------------------------------------------------
# If Block 1 has already been run, this reuses its DEFAULT_DATA_DIR. Otherwise
# it falls back to the same Endovascular/Jill example path.
try:
    FILTER_DEFAULT_DATA_DIR = DEFAULT_DATA_DIR
except NameError:
    FILTER_DEFAULT_DATA_DIR = Path(
        "/Users/jf/Library/CloudStorage/SynologyDrive-Endovascular/Jill/"
        "20260617 stimulation Pt/"
        "1_260617_180931 anodic-lead 100us 50uA 5 pulses 1000Hz"
    )


def folder_path_from_text(value: str) -> Path:
    """Clean a pasted folder path from the widget and return an absolute Path."""
    cleaned = value.strip().strip('"').strip("'")
    cleaned = cleaned.replace("\r", "").replace("\n", "")
    return Path(cleaned).expanduser().resolve()


# -----------------------------------------------------------------------------
# 4) Build the filtered plot controls.
# -----------------------------------------------------------------------------
filter_folder_text = widgets.Textarea(
    value=str(FILTER_DEFAULT_DATA_DIR),
    description="RHS folder",
    placeholder="Paste RHS data folder path here",
    continuous_update=False,
    layout=widgets.Layout(width="100%", height="46px"),
    style={"description_width": "90px"},
)
filter_channel_text = widgets.Text(
    value="A-014",
    description="Channels",
    placeholder="A-014 or A-014-16",
    layout=widgets.Layout(width="240px"),
    style={"description_width": "75px"},
)
bandpass_text = widgets.Text(
    value="all",
    description="Bandpass",
    placeholder="all or 200-400 Hz",
    layout=widgets.Layout(width="230px"),
    style={"description_width": "80px"},
)
amplitude_text = widgets.Text(
    value="-100 - 100 uV",
    description="Amplitude",
    layout=widgets.Layout(width="230px"),
    style={"description_width": "85px"},
)
filter_time_text = widgets.Text(
    value="all",
    description="Time (s)",
    placeholder="all or 10-20",
    layout=widgets.Layout(width="190px"),
    style={"description_width": "70px"},
)
filter_max_points_int = widgets.IntText(
    value=600_000,
    description="Max points",
    layout=widgets.Layout(width="210px"),
    style={"description_width": "90px"},
)
filter_generate_button = widgets.Button(
    description="Generate Preview",
    icon="play",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)
filter_save_button = widgets.Button(
    description="Save PNG",
    icon="save",
    button_style="success",
    disabled=True,
    layout=widgets.Layout(width="110px"),
)
filter_status = widgets.HTML(value="Choose filter settings, then click <b>Generate Preview</b>.")
filter_target_label = widgets.HTML(value="<b>Target:</b> no preview generated yet")
filter_preview_output = widgets.Output()
filter_current_preview = {"png": None, "output_path": None}

display(
    widgets.VBox(
        [
            filter_folder_text,
            widgets.HBox([filter_channel_text, bandpass_text, amplitude_text]),
            widgets.HBox([filter_time_text, filter_max_points_int]),
            widgets.HBox([filter_generate_button, filter_save_button, filter_target_label]),
            filter_status,
            filter_preview_output,
        ]
    )
)


# -----------------------------------------------------------------------------
# 5) Generate and save the bandpass filtered preview.
# -----------------------------------------------------------------------------
def generate_filtered_preview(_button=None) -> None:
    """Read selected RHS folder, apply filters, and show inline preview."""
    filter_save_button.disabled = True
    filter_current_preview["png"] = None
    filter_current_preview["output_path"] = None
    filter_target_label.value = "<b>Target:</b> no preview generated yet"
    filter_preview_output.clear_output()

    data_folder = folder_path_from_text(filter_folder_text.value)
    max_points = int(filter_max_points_int.value)

    try:
        channels = parse_channel_selection(filter_channel_text.value)
        band_hz = parse_frequency_range(bandpass_text.value)
        amplitude_uV = parse_amplitude_range(amplitude_text.value)
        time_window = parse_time_window(filter_time_text.value)
    except ValueError as exc:
        filter_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    output_label = channel_selection_label(channels)
    if time_window is not None:
        output_label = f"{output_label}_{time_window_label(time_window)}"
    output_path = default_filtered_output_path(data_folder, output_label, band_hz, amplitude_uV)

    if not data_folder.exists():
        filter_status.value = f"<b style='color:#b00020'>Folder not found:</b> {data_folder}"
        return
    if not list(data_folder.glob("*.rhs")):
        filter_status.value = f"<b style='color:#b00020'>No .rhs files found in:</b> {data_folder}"
        return

    filter_status.value = f"Reading RHS files and {format_bandpass_status(band_hz)}..."
    channel_data = []
    stim_channel_data = []
    sample_rate_hz = None
    loaded = None
    try:
        for channel in channels:
            raw_uV, stim_uA, channel_sample_rate_hz, channel_loaded = read_rhs_folder(data_folder, channel)
            if sample_rate_hz is not None and channel_sample_rate_hz != sample_rate_hz:
                raise ValueError("Selected channels have different sample rates.")
            sample_rate_hz = channel_sample_rate_hz
            loaded = channel_loaded
            channel_data.append((channel, raw_uV))
            stim_channel_data.append((channel, raw_uV, stim_uA))
    except (FileNotFoundError, ValueError) as exc:
        filter_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    stim_channel_info = find_stim_channel_in_data(stim_channel_data, slice(0, stim_channel_data[0][1].size))
    stim_channel_name = stim_channel_info[0] if stim_channel_info is not None else None

    try:
        fig, summaries = plot_filtered_channels(
            channel_data=channel_data,
            sample_rate_hz=sample_rate_hz,
            folder=data_folder,
            band_hz=band_hz,
            amplitude_uV=amplitude_uV,
            max_points=max_points,
            time_window=time_window,
            stim_channel_name=stim_channel_name,
        )
    except ValueError as exc:
        filter_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    preview_buffer = BytesIO()
    fig.savefig(preview_buffer, format="png", dpi=220)
    plt.close(fig)
    preview_png = preview_buffer.getvalue()

    filter_current_preview["png"] = preview_png
    filter_current_preview["output_path"] = output_path
    filter_save_button.disabled = False
    filter_target_label.value = f"<b>Target:</b> {output_path}"

    rms_text = ", ".join(f"{item['channel_name']}: {item['filtered_rms_uV']:.2f} uV" for item in summaries)
    samples_selected = sum(int(item['samples_in_amplitude_range']) for item in summaries)
    samples_total = sum(int(item['samples_total']) for item in summaries)
    percent_selected = samples_selected / samples_total * 100.0 if samples_total else 0.0
    time_status = "all time" if time_window is None else f"{time_window[0]:g}-{time_window[1]:g} s"
    stim_status = "no nonzero stim_data found in selected channels" if stim_channel_name is None else f"stim channel: {stim_channel_name} *"
    filter_status.value = (
        f"Loaded {len(loaded)} RHS file(s) for {len(channels)} channel(s). "
        f"Time window: {time_status}. "
        f"{stim_status}. "
        f"Signal RMS: {rms_text}. "
        f"Amplitude-selected samples: {samples_selected} "
        f"({percent_selected:.3f}%)."
    )

    with filter_preview_output:
        display(widgets.Image(value=preview_png, format="png", layout=widgets.Layout(width="100%")))


def save_filtered_png(_button=None) -> None:
    """Save the current filtered preview into the selected RHS folder only."""
    preview_png = filter_current_preview.get("png")
    output_path = filter_current_preview.get("output_path")
    if preview_png is None or output_path is None:
        filter_status.value = "<b style='color:#b00020'>Generate a preview before saving.</b>"
        return
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = output_path.with_name(f".{output_path.stem}.tmp{output_path.suffix}")
    temp_path.write_bytes(preview_png)
    os.replace(temp_path, output_path)
    filter_target_label.value = f"<b>Saved:</b> {output_path}"
    filter_status.value = f"Saved PNG inside selected data folder: <b>{output_path.parent}</b>"


filter_generate_button.on_click(generate_filtered_preview)
filter_save_button.on_click(save_filtered_png)


# 3. Plot Stim-Triggered Bandpass Events

Use this third block to split the session into stim-triggered events based on RHS `stim_data` onsets, then place all events into one combined PNG. Each event panel starts at one stim command onset and ends immediately before the next stim command onset, unless you enter an event-relative time window such as `0-1s`. Events are arranged 3 per row so each row stays compact.

Function 3 uses the same filters as Function 2: channels, Bandpass (`all` or a numeric range), signed amplitude window, event-relative time window, and max points. Stim onsets are detected from the selected channel that contains nonzero `stim_data`. `Train gap (ms)` controls how close stim pulses can be while still being grouped as one train/event.

The preview is not saved until you click **Save PNG**.

### Helper code used by Block 3

This block uses [`plot_rhs_stim_triggered_events.py`](plot_rhs_stim_triggered_events.py) for stim-triggered event detection and plotting.

- `build_stim_triggered_events(...)` groups nearby `stim_data` pulses into train/event onsets, then creates event windows from each onset to immediately before the next onset.
- `filter_channel_data(...)` keeps all recorded frequencies when Bandpass is `all`, or bandpass filters each selected channel once before event slicing.
- `plot_stim_triggered_events_grid(...)` creates one combined grid plot with 3 stim events per row, and adds the matching `stim_data` waveform below the selected response channels in each event panel.
- `default_stim_events_grid_output_path(...)` names the single combined event PNG.

In [ ]:
from __future__ import annotations

import os
import tempfile
from io import BytesIO
from pathlib import Path

# -----------------------------------------------------------------------------
# 1) Configure notebook output so plots are embedded, not pop-up windows.
# -----------------------------------------------------------------------------
_cache_root = Path(tempfile.gettempdir()) / "codex_matplotlib_cache"
_mpl_cache = _cache_root / "mpl"
_xdg_cache = _cache_root / "xdg"
_mpl_cache.mkdir(parents=True, exist_ok=True)
_xdg_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_mpl_cache))
os.environ.setdefault("XDG_CACHE_HOME", str(_xdg_cache))
os.environ["MPLBACKEND"] = "Agg"

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# -----------------------------------------------------------------------------
# 2) Import channel/time parsers, RHS reader, and stim-triggered helpers.
# -----------------------------------------------------------------------------
from plot_rhs_raw_wideband_with_stim_legend import (
    channel_selection_label,
    find_stim_channel_in_data,
    parse_channel_selection,
    parse_time_window,
    read_rhs_folder,
    time_window_label,
)
from plot_rhs_filtered_wideband import format_bandpass_status, parse_amplitude_range, parse_frequency_range
from plot_rhs_stim_triggered_events import (
    build_stim_triggered_events,
    default_stim_events_grid_output_path,
    filter_channel_data,
    plot_stim_triggered_events_grid,
)


# -----------------------------------------------------------------------------
# 3) Choose the default data folder and stim-triggered plot settings.
# -----------------------------------------------------------------------------
try:
    EVENT_DEFAULT_DATA_DIR = DEFAULT_DATA_DIR
except NameError:
    EVENT_DEFAULT_DATA_DIR = Path(
        "/Users/jf/Library/CloudStorage/SynologyDrive-Endovascular/Jill/"
        "20260617 stimulation Pt/"
        "1_260617_180931 anodic-lead 100us 50uA 5 pulses 1000Hz"
    )


def folder_path_from_text(value: str) -> Path:
    """Clean a pasted folder path from the widget and return an absolute Path."""
    cleaned = value.strip().strip('"').strip("'")
    cleaned = cleaned.replace("\r", "").replace("\n", "")
    return Path(cleaned).expanduser().resolve()


# -----------------------------------------------------------------------------
# 4) Build the stim-triggered event controls.
# -----------------------------------------------------------------------------
event_folder_text = widgets.Textarea(
    value=str(EVENT_DEFAULT_DATA_DIR),
    description="RHS folder",
    placeholder="Paste RHS data folder path here",
    continuous_update=False,
    layout=widgets.Layout(width="100%", height="46px"),
    style={"description_width": "90px"},
)
event_channel_text = widgets.Text(
    value="A-014",
    description="Channels",
    placeholder="A-014 or A-014-16",
    layout=widgets.Layout(width="240px"),
    style={"description_width": "75px"},
)
event_bandpass_text = widgets.Text(
    value="all",
    description="Bandpass",
    placeholder="all or 200-400 Hz",
    layout=widgets.Layout(width="230px"),
    style={"description_width": "80px"},
)
event_amplitude_text = widgets.Text(
    value="-100 - 100 uV",
    description="Amplitude",
    layout=widgets.Layout(width="230px"),
    style={"description_width": "85px"},
)
event_time_text = widgets.Text(
    value="all",
    description="Event time (s)",
    placeholder="all or 0-1",
    layout=widgets.Layout(width="220px"),
    style={"description_width": "105px"},
)
event_train_gap_float = widgets.FloatText(
    value=10.0,
    description="Train gap (ms)",
    layout=widgets.Layout(width="230px"),
    style={"description_width": "105px"},
)
event_max_points_int = widgets.IntText(
    value=200_000,
    description="Max points",
    layout=widgets.Layout(width="210px"),
    style={"description_width": "90px"},
)
event_generate_button = widgets.Button(
    description="Generate Event Previews",
    icon="play",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)
event_save_button = widgets.Button(
    description="Save PNG",
    icon="save",
    button_style="success",
    disabled=True,
    layout=widgets.Layout(width="120px"),
)
event_status = widgets.HTML(value="Choose event settings, then click <b>Generate Event Previews</b>.")
event_target_label = widgets.HTML(value="<b>Target:</b> no previews generated yet")
event_preview_output = widgets.Output()
event_current_preview = {"png": None, "output_path": None, "events": []}

display(
    widgets.VBox(
        [
            event_folder_text,
            widgets.HBox([event_channel_text, event_bandpass_text, event_amplitude_text]),
            widgets.HBox([event_time_text, event_train_gap_float, event_max_points_int]),
            widgets.HBox([event_generate_button, event_save_button, event_target_label]),
            event_status,
            event_preview_output,
        ]
    )
)


# -----------------------------------------------------------------------------
# 5) Generate one combined stim-triggered event preview and save on request.
# -----------------------------------------------------------------------------
def generate_event_previews(_button=None) -> None:
    """Read RHS data, detect stim onsets, and show one combined inline plot."""
    event_save_button.disabled = True
    event_current_preview["png"] = None
    event_current_preview["output_path"] = None
    event_current_preview["events"] = []
    event_target_label.value = "<b>Target:</b> no previews generated yet"
    event_preview_output.clear_output()

    data_folder = folder_path_from_text(event_folder_text.value)
    max_points = int(event_max_points_int.value)
    train_gap_ms = float(event_train_gap_float.value)
    if train_gap_ms < 0:
        event_status.value = "<b style='color:#b00020'>Train gap must be 0 ms or greater.</b>"
        return

    try:
        channels = parse_channel_selection(event_channel_text.value)
        band_hz = parse_frequency_range(event_bandpass_text.value)
        amplitude_uV = parse_amplitude_range(event_amplitude_text.value)
        time_window = parse_time_window(event_time_text.value)
    except ValueError as exc:
        event_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    if not data_folder.exists():
        event_status.value = f"<b style='color:#b00020'>Folder not found:</b> {data_folder}"
        return
    if not list(data_folder.glob("*.rhs")):
        event_status.value = f"<b style='color:#b00020'>No .rhs files found in:</b> {data_folder}"
        return

    event_status.value = f"Reading RHS files from <b>{data_folder}</b>..."
    raw_channel_data = []
    stim_channel_data = []
    sample_rate_hz = None
    loaded = None
    try:
        for channel in channels:
            raw_uV, stim_uA, channel_sample_rate_hz, channel_loaded = read_rhs_folder(data_folder, channel)
            if sample_rate_hz is not None and channel_sample_rate_hz != sample_rate_hz:
                raise ValueError("Selected channels have different sample rates.")
            sample_rate_hz = channel_sample_rate_hz
            loaded = channel_loaded
            raw_channel_data.append((channel, raw_uV))
            stim_channel_data.append((channel, raw_uV, stim_uA))
    except (FileNotFoundError, ValueError) as exc:
        event_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    stim_channel_info = find_stim_channel_in_data(stim_channel_data, slice(0, stim_channel_data[0][1].size))
    if stim_channel_info is None:
        event_status.value = "<b style='color:#b00020'>No nonzero stim_data found in selected channels.</b>"
        return
    stim_detection_channel, stim_uA_for_events, _pulse_segments = stim_channel_info

    try:
        events = build_stim_triggered_events(
            stim_uA_for_events,
            sample_rate_hz,
            None,
            merge_gap_ms=train_gap_ms,
        )
    except ValueError as exc:
        event_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    if not events:
        event_status.value = (
            f"<b style='color:#b00020'>No stim_data onsets found on {stim_detection_channel}.</b>"
        )
        return

    event_status.value = (
        f"Detected {len(events)} stim event(s) on {stim_detection_channel} "
        f"with train gap {train_gap_ms:g} ms; "
        f"{format_bandpass_status(band_hz)}..."
    )
    try:
        filtered_channel_data = filter_channel_data(raw_channel_data, sample_rate_hz, band_hz)
    except ValueError as exc:
        event_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    channel_label = channel_selection_label(channels)
    time_label = time_window_label(time_window)
    output_path = default_stim_events_grid_output_path(
        folder=data_folder,
        channel_label=channel_label,
        band_hz=band_hz,
        amplitude_uV=amplitude_uV,
        time_label=time_label,
        event_count=len(events),
    )

    try:
        fig, summaries = plot_stim_triggered_events_grid(
            filtered_channel_data=filtered_channel_data,
            sample_rate_hz=sample_rate_hz,
            folder=data_folder,
            events=events,
            band_hz=band_hz,
            amplitude_uV=amplitude_uV,
            max_points=max_points,
            events_per_row=3,
            stim_channel_name=stim_detection_channel,
            stim_uA=stim_uA_for_events,
            event_time_window=time_window,
        )
    except ValueError as exc:
        event_status.value = f"<b style='color:#b00020'>{exc}</b>"
        return

    preview_buffer = BytesIO()
    fig.savefig(preview_buffer, format="png", dpi=220)
    plt.close(fig)
    preview_png = preview_buffer.getvalue()

    event_current_preview["png"] = preview_png
    event_current_preview["output_path"] = output_path
    event_current_preview["events"] = events
    event_save_button.disabled = False
    event_target_label.value = f"<b>Target:</b> {output_path}"

    with event_preview_output:
        display(widgets.HTML(
            value=(
                f"<b>{len(events)} event(s) in one PNG</b>, 3 events per row; "
                f"target {output_path.name}"
            )
        ))
        display(widgets.Image(value=preview_png, format="png", layout=widgets.Layout(width="100%")))

    _summary_count = len(summaries)
    first_onset = events[0].onset_s
    last_onset = events[-1].onset_s
    event_status.value = (
        f"Generated one combined PNG preview with {len(events)} event(s) "
        f"from {len(loaded)} RHS file(s), "
        f"using {stim_detection_channel} stim_data for grouped onsets "
        f"({first_onset:g} to {last_onset:g} s; train gap {train_gap_ms:g} ms; "
        f"event window {time_window_label(time_window)})."
    )


def save_event_pngs(_button=None) -> None:
    """Save the combined stim-triggered event preview into the selected data folder."""
    preview_png = event_current_preview.get("png")
    output_path = event_current_preview.get("output_path")
    events = event_current_preview.get("events", [])
    if preview_png is None or output_path is None:
        event_status.value = "<b style='color:#b00020'>Generate an event preview before saving.</b>"
        return

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = output_path.with_name(f".{output_path.stem}.tmp{output_path.suffix}")
    temp_path.write_bytes(preview_png)
    os.replace(temp_path, output_path)

    event_target_label.value = f"<b>Saved:</b> {output_path}"
    event_status.value = (
        f"Saved one combined PNG with {len(events)} event(s) inside selected data folder: "
        f"<b>{output_path.parent}</b>"
    )


event_generate_button.on_click(generate_event_previews)
event_save_button.on_click(save_event_pngs)


Def:
train gap (ms) controls how close stim pulses can be while still being grouped as one stimulation train/event. For example, for a 100 Hz train, pulses are about 10 ms apart. With Train gap (ms) = 12, those 5 pulses are grouped into one event. In Function 3, Time (s) is relative to each stim onset, so 0-1s shows the first second after every event onset.